In [1]:
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [11]:
import sys
from pathlib import Path
from math import cos, sin, pi, radians

# Make sure ur3e-control is on the path
sys.path.insert(0, str(Path.cwd().parents[1]))

from my_simulation.iscoin_sim.kinematics import (
    analytical_ik,
    forward_kinematics_matrix,
    pose_to_matrix,
    matrix_to_tcp6d,
)

from my_simulation.iscoin_sim import ISCoinSim as ISCoin
from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor


# # Connect to the simulator
iscoin = ISCoin()
robot = iscoin.robot_control
print("Connected to simulator")

ConnectionError: Container 'iscoin_simulator' is not running. Start the simulator first:
  cd ur3e-simulator/.docker && docker compose run --rm cpu
  ros2 launch iscoin_simulation_gz iscoin_sim_control.launch.py

In [12]:
# Go to home position
home = Joint6D.createFromRadians(1.1859, -1.4452, 1.2389, -1.3639, -1.5693, -0.3849)
robot.movej(home, a=radians(80), v=radians(60))

tcp_home = robot.get_actual_tcp_pose()
print(f"Home TCP: {tcp_home}")

NameError: name 'robot' is not defined

In [13]:
target_tcp = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.35,     # y
      0.20,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )

In [2]:
import trimesh
print(f"trimesh version: {trimesh.__version__}")

trimesh version: 4.11.2


In [15]:
mesh_path = "../3d_objects/duckround.stl"

In [3]:
mesh_path = "../3d_objects/trapez.stl"

In [4]:
obstacle = trimesh.load(mesh_path)
print(f"Loaded: {mesh_path}")
print(f"Type: {type(obstacle)}")

Loaded: ../3d_objects/trapez.stl
Type: <class 'trimesh.base.Trimesh'>


In [5]:
print(f"Vertices: {len(obstacle.vertices)}")
print(f"Faces:    {len(obstacle.faces)}")

Vertices: 12
Faces:    20


In [6]:
print(f"Watertight: {obstacle.is_watertight}")

if not obstacle.is_watertight:
    print("Mesh is not watertight — attempting repair...")
    trimesh.repair.fix_winding(obstacle)
    trimesh.repair.fill_holes(obstacle)
    trimesh.repair.fix_normals(obstacle)
    print(f"Watertight after repair: {obstacle.is_watertight}")

if not obstacle.is_watertight:
    print("Repair failed — falling back to convex hull")
    obstacle = obstacle.convex_hull
    print(f"Watertight (convex hull): {obstacle.is_watertight}")

Watertight: True


In [7]:
bbox_min, bbox_max = obstacle.bounds
print(f"Bounding box min: {bbox_min}")
print(f"Bounding box max: {bbox_max}")
print(f"Size: {bbox_max - bbox_min}")

Bounding box min: [-60.         -40.         -10.00000286]
Bounding box max: [60.         40.         35.75260925]
Size: [120.          80.          45.75261211]


In [8]:
print(f"Centroid: {obstacle.centroid}")

Centroid: [-5.03618369e-16  8.48922306e-07  5.97681579e+00]


In [9]:
# Use the centroid — should always be inside a convex mesh
test_inside = obstacle.centroid
result = obstacle.contains(np.array([test_inside]))
print(f"Point: {test_inside}")
print(f"Inside: {result[0]}")

Point: [-5.03618369e-16  8.48922306e-07  5.97681579e+00]
Inside: True


In [10]:
# Point far from the mesh
test_outside = bbox_max + 10.0
result = obstacle.contains(np.array([test_outside]))
print(f"Point: {test_outside}")
print(f"Inside: {result[0]}")

Point: [70.         50.         45.75260925]
Inside: False


In [11]:
def is_inside(mesh, point):
    """Return True if point is inside the mesh."""
    return mesh.contains(np.array([point]))[0]

# Quick sanity check
print(f"Centroid inside: {is_inside(obstacle, obstacle.centroid)}")

Centroid inside: True


In [12]:
rng = np.random.default_rng(42)
points = rng.uniform(bbox_min - 0.05, bbox_max + 0.05, size=(200, 3))
print(f"Generated {len(points)} random points")
print(f"Range: {points.min(axis=0)} to {points.max(axis=0)}")

Generated 200 random points
Range: [-59.1715338  -39.06455486  -9.80103078] to [59.13430524 39.97828891 35.66664057]


In [13]:
inside_mask = obstacle.contains(points)
print(f"Inside:  {inside_mask.sum()}")
print(f"Outside: {(~inside_mask).sum()}")

Inside:  135
Outside: 65


In [14]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot_trisurf(
    obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
    triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Obstacle mesh')
ax.set_box_aspect([1, 1, 1])
mid = (bbox_max + bbox_min) / 2
half = (bbox_max - bbox_min).max()  # double the space
ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(0, 2 * half)
plt.tight_layout()
plt.show()

In [15]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot_trisurf(
    obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
    triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
)

ax.scatter(*points[inside_mask].T, c='red', s=15, label='Inside', depthshade=True)
ax.scatter(*points[~inside_mask].T, c='blue', s=5, alpha=0.3, label='Outside', depthshade=True)

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Point containment test')
ax.legend()
ax.set_box_aspect([1, 1, 1])
mid = (bbox_max + bbox_min) / 2
half = (bbox_max - bbox_min).max()
ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(0, 2 * half)
plt.tight_layout()
plt.show()

In [16]:
def segment_collides(A, B, mesh):
    """Check if straight-line segment A->B collides with the mesh.
    Collision = any point on the segment is inside or crosses the mesh surface.
    """
    # If either endpoint is inside, it's a collision
    if mesh.contains(np.array([A, B])).any():
        return True
    # Check if the segment crosses the mesh surface
    direction = B - A
    length = np.linalg.norm(direction)
    if length < 1e-10:
        return False
    d_unit = direction / length
    locations, _, _ = mesh.ray.intersects_location(
        ray_origins=np.array([A]), ray_directions=np.array([d_unit])
    )
    if len(locations) == 0:
        return False
    return bool(np.any(np.linalg.norm(locations - A, axis=1) <= length))

print("segment_collides() defined")

segment_collides() defined


In [17]:
A_hit = bbox_min - 0.1
B_hit = obstacle.centroid
print(f"A = {A_hit}")
print(f"B = {B_hit}")
print(f"Collides: {segment_collides(A_hit, B_hit, obstacle)}")

A = [-60.1        -40.1        -10.10000286]
B = [-5.03618369e-16  8.48922306e-07  5.97681579e+00]
Collides: True


In [18]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot_trisurf(
    obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
    triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
)

# Draw the hitting segment
ax.plot(*np.array([A_hit, B_hit]).T, c='red', linewidth=2, label='Segment (hit)')
ax.scatter(*A_hit, c='red', s=50, marker='^', zorder=5)
ax.scatter(*B_hit, c='red', s=50, marker='o', zorder=5)

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title(f'Segment collision — Collides: {segment_collides(A_hit, B_hit, obstacle)}')
ax.legend()
ax.set_box_aspect([1, 1, 1])
all_pts = np.vstack([obstacle.vertices, [A_hit], [B_hit]])
lo, hi = all_pts.min(axis=0), all_pts.max(axis=0)
mid = (hi + lo) / 2
half = (hi - lo).max()
ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(0, 2 * half)
plt.tight_layout()
plt.show()

In [19]:
A_miss = bbox_max + np.array([0.1, 0.1, 0.0])
B_miss = bbox_max + np.array([0.1, 0.1, 1.0])
print(f"A = {A_miss}")
print(f"B = {B_miss}")
print(f"Collides: {segment_collides(A_miss, B_miss, obstacle)}")

A = [60.1        40.1        35.75260925]
B = [60.1        40.1        36.75260925]
Collides: False


In [20]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot_trisurf(
    obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
    triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
)

# Draw the missing segment
ax.plot(*np.array([A_miss, B_miss]).T, c='green', linewidth=2, label='Segment (miss)')
ax.scatter(*A_miss, c='green', s=50, marker='^', zorder=5)
ax.scatter(*B_miss, c='green', s=50, marker='o', zorder=5)

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title(f'Segment collision — Collides: {segment_collides(A_miss, B_miss, obstacle)}')
ax.legend()
ax.set_box_aspect([1, 1, 1])
all_pts = np.vstack([obstacle.vertices, [A_miss], [B_miss]])
lo, hi = all_pts.min(axis=0), all_pts.max(axis=0)
mid = (hi + lo) / 2
half = (hi - lo).max()
ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(0, 2 * half)
plt.tight_layout()
plt.show()

In [28]:
# === Collision detection test suite ===
# All segment cases on one plot: red = collision, green = no collision

centroid = obstacle.centroid
mid_bbox = (bbox_max + bbox_min) / 2

test_segments = [
    # Case 1: Outside → inside (endpoint B inside)
    ("Outside → Inside",       bbox_min - 10.0,              centroid),
    # Case 2: Inside → outside (endpoint A inside)
    ("Inside → Outside",       centroid,                     bbox_max + 10.0),
    # Case 3: Both inside (fully embedded)
    ("Both Inside",            centroid - 1.0,               centroid + 1.0),
    # Case 4: Outside → outside, passes through (pierces mesh)
    ("Through mesh",           bbox_min - 5.0,               bbox_max + 5.0),
    # Case 5: Completely outside, no intersection (above)
    ("Miss above",             bbox_max + np.array([0, 0, 5]),   bbox_max + np.array([50, 50, 5])),
    # Case 6: Completely outside, no intersection (side)
    ("Miss side",              bbox_max + np.array([10, 0, 0]),  bbox_max + np.array([10, 50, 0])),
    # Case 7: Grazing — segment runs along the edge, parallel and outside
    ("Miss parallel",          np.array([bbox_min[0] - 5, bbox_min[1], 0]),
                               np.array([bbox_max[0] + 5, bbox_min[1], 0])),
    # Case 8: Short segment far away
    ("Miss far away",          bbox_max + 100.0,             bbox_max + 101.0),
]

# Run all tests
print(f"{'Case':<22} {'Collides':>8}")
print("-" * 32)
results = []
for name, A, B in test_segments:
    col = segment_collides(A, B, obstacle)
    results.append((name, np.array(A, dtype=float), np.array(B, dtype=float), col))
    print(f"{name:<22} {str(col):>8}")

# Plot all segments on one figure
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

ax.plot_trisurf(
    obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
    triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
)

for name, A, B, col in results:
    color = 'red' if col else 'green'
    ax.plot(*np.array([A, B]).T, c=color, linewidth=2)
    ax.scatter(*A, c=color, s=40, marker='^', zorder=5)
    ax.scatter(*B, c=color, s=40, marker='o', zorder=5)
    label_pos = (A + B) / 2
    ax.text(label_pos[0], label_pos[1], label_pos[2], name, fontsize=7, color=color)

# Dummy lines for legend
ax.plot([], [], c='red', linewidth=2, label='Collision')
ax.plot([], [], c='green', linewidth=2, label='No collision')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Segment collision test suite')
ax.legend(fontsize=10)
ax.set_box_aspect([1, 1, 1])
all_pts = np.vstack([r[1] for r in results] + [r[2] for r in results] + [obstacle.vertices])
lo, hi = all_pts.min(axis=0), all_pts.max(axis=0)
view_mid = (hi + lo) / 2
view_half = (hi - lo).max() / 2 * 1.1
ax.set_xlim(view_mid[0] - view_half, view_mid[0] + view_half)
ax.set_ylim(view_mid[1] - view_half, view_mid[1] + view_half)
ax.set_zlim(0, 2 * view_half)
plt.tight_layout()
plt.show()

Case                   Collides
--------------------------------
Outside → Inside           True
Inside → Outside           True
Both Inside                True
Through mesh               True
Miss above                False
Miss side                 False
Miss parallel              True
Miss far away             False


In [22]:
# === Pathfinding parameters ===
start = np.array([bbox_min[0] - 20.0, 0.0, 20.0])
goal  = np.array([bbox_max[0] + 20.0, 0.0, 20.0])

safe_margin = 5.0   # minimum clearance from the mesh surface
push_step   = 5.0    # how far to push per iteration when escaping the mesh
max_depth   = 10     # max recursion depth to prevent infinite splitting

print(f"Start:       {start}")
print(f"Goal:        {goal}")
print(f"Safe margin: {safe_margin}")
print(f"Push step:   {push_step}")
print(f"Max depth:   {max_depth}")

Start:       [-80.   0.  20.]
Goal:        [80.  0. 20.]
Safe margin: 5.0
Push step:   5.0
Max depth:   10


In [23]:
# === Helper: perpendicular direction with max Z component ===
def perpendicular_up(A, B):
    """Return a unit vector perpendicular to A->B that points as much upward (+Z) as possible."""
    AB = B - A
    AB_norm = AB / np.linalg.norm(AB)

    # Project Z-hat onto the plane perpendicular to AB
    z_hat = np.array([0.0, 0.0, 1.0])
    perp = z_hat - np.dot(z_hat, AB_norm) * AB_norm

    if np.linalg.norm(perp) < 1e-10:
        # AB is vertical — fall back to Y direction
        y_hat = np.array([0.0, 1.0, 0.0])
        perp = y_hat - np.dot(y_hat, AB_norm) * AB_norm

    return perp / np.linalg.norm(perp)

# Quick test
d = perpendicular_up(start, goal)
print(f"Push direction: {d}")
print(f"Dot with AB:    {np.dot(d, goal - start):.10f} (should be ~0)")
print(f"Z component:    {d[2]:.4f} (should be high)")

Push direction: [0. 0. 1.]
Dot with AB:    0.0000000000 (should be ~0)
Z component:    1.0000 (should be high)


In [24]:
def closest_distance_to_mesh(point, mesh):
    """Return the shortest distance from a point to the mesh surface."""
    closest_point, distance, _ = trimesh.proximity.closest_point(mesh, np.array([point]))
    return float(distance[0])


def lift_midpoint(A, B, mesh, safe_margin, push_step, max_push=500.0):
    """Take the midpoint of A->B and push it upward until it is safe_margin away from the mesh.

    Returns the lifted point, or None if max_push is exceeded.
    """
    M = (A + B) / 2.0
    up = perpendicular_up(A, B)

    offset = 0.0
    while offset <= max_push:
        candidate = M + up * offset
        dist = closest_distance_to_mesh(candidate, mesh)
        inside = is_inside(mesh, candidate)
        if not inside and dist >= safe_margin:
            return candidate
        offset += push_step

    return None  # could not find a safe point


# Quick test: lift the midpoint between start and goal
lifted = lift_midpoint(start, goal, obstacle, safe_margin, push_step)
print(f"Start:          {start}")
print(f"Goal:           {goal}")
print(f"Lifted midpoint: {lifted}")
print(f"Distance to mesh: {closest_distance_to_mesh(lifted, obstacle):.2f} (need >= {safe_margin})")

Start:          [-80.   0.  20.]
Goal:           [80.  0. 20.]
Lifted midpoint: [ 0.  0. 45.]
Distance to mesh: 9.25 (need >= 5.0)


In [25]:
def find_path(A, B, mesh, safe_margin, push_step, max_depth, _depth=0):
    """Recursive midpoint-lifting pathfinder.

    Returns a list of waypoints [A, ..., B] that form a collision-free path.
    """
    # Base case: no collision on this segment — we're done
    if not segment_collides(A, B, mesh):
        return [A, B]

    # Safety: stop recursing if we hit max depth
    if _depth >= max_depth:
        print(f"  [depth {_depth}] max depth reached, keeping segment as-is")
        return [A, B]

    # Lift the midpoint until it's safe
    M = lift_midpoint(A, B, mesh, safe_margin, push_step)
    if M is None:
        print(f"  [depth {_depth}] WARNING: could not lift midpoint, keeping segment")
        return [A, B]

    print(f"  [depth {_depth}] lifted midpoint to {M}")

    # Recurse on each half: A->M and M->B
    left_path  = find_path(A, M, mesh, safe_margin, push_step, max_depth, _depth + 1)
    right_path = find_path(M, B, mesh, safe_margin, push_step, max_depth, _depth + 1)

    # Merge: left_path ends with M, right_path starts with M — avoid duplicate
    return left_path + right_path[1:]


# Run the pathfinder
print("Finding path...")
path = find_path(start, goal, obstacle, safe_margin, push_step, max_depth)
print(f"\nPath found with {len(path)} waypoints:")
for i, wp in enumerate(path):
    print(f"  [{i}] {wp}")

Finding path...
  [depth 0] lifted midpoint to [ 0.  0. 45.]
  [depth 1] lifted midpoint to [-40.    0.   32.5]
  [depth 2] lifted midpoint to [-21.49137497   0.          43.52239989]
  [depth 1] lifted midpoint to [40.   0.  32.5]
  [depth 2] lifted midpoint to [21.49137497  0.         43.52239989]

Path found with 7 waypoints:
  [0] [-80.   0.  20.]
  [1] [-40.    0.   32.5]
  [2] [-21.49137497   0.          43.52239989]
  [3] [ 0.  0. 45.]
  [4] [21.49137497  0.         43.52239989]
  [5] [40.   0.  32.5]
  [6] [80.  0. 20.]


In [26]:
# === Verify: check every segment of the final path for collisions ===
print("Verifying final path...")
all_clear = True
for i in range(len(path) - 1):
    col = segment_collides(path[i], path[i+1], obstacle)
    status = "COLLISION" if col else "ok"
    print(f"  Segment {i}->{i+1}: {status}")
    if col:
        all_clear = False

print(f"\nAll segments collision-free: {all_clear}")

Verifying final path...
  Segment 0->1: ok
  Segment 1->2: ok
  Segment 2->3: ok
  Segment 3->4: ok
  Segment 4->5: ok
  Segment 5->6: ok

All segments collision-free: True


In [27]:
# === Visualize the final path (with azimuth + elevation sliders) ===
from matplotlib.widgets import Slider

path_arr = np.array(path)

fig = plt.figure(figsize=(12, 9))
fig.subplots_adjust(bottom=0.20)
ax = fig.add_subplot(111, projection='3d')

# Draw the obstacle
ax.plot_trisurf(
    obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
    triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
)

# Draw path segments
for i in range(len(path) - 1):
    seg = np.array([path[i], path[i+1]])
    ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], c='blue', linewidth=2)

# Draw waypoints
ax.scatter(*path_arr.T, c='blue', s=60, zorder=5, label='Waypoints')
ax.scatter(*start, c='green', s=120, marker='^', zorder=6, label='Start')
ax.scatter(*goal,  c='red',   s=120, marker='*', zorder=6, label='Goal')

# Label waypoints
for i, wp in enumerate(path):
    ax.text(wp[0], wp[1], wp[2] + 3, str(i), fontsize=8, ha='center', color='blue')

# Draw the direct (colliding) path for comparison
ax.plot([start[0], goal[0]], [start[1], goal[1]], [start[2], goal[2]],
        c='red', linewidth=1, linestyle='--', alpha=0.4, label='Direct (collides)')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title(f'Pathfinding result — {len(path)} waypoints')
ax.legend()
ax.set_box_aspect([1, 1, 1])
all_pts = np.vstack([obstacle.vertices, path_arr])
lo, hi = all_pts.min(axis=0), all_pts.max(axis=0)
view_mid = (hi + lo) / 2
view_half = (hi - lo).max() / 2 * 1.2
ax.set_xlim(view_mid[0] - view_half, view_mid[0] + view_half)
ax.set_ylim(view_mid[1] - view_half, view_mid[1] + view_half)
ax.set_zlim(0, 2 * view_half)

# Initial view
ax.view_init(elev=25, azim=-60)
ax.disable_mouse_rotation()

# Azimuth slider (horizontal rotation around Z)
azim_ax = fig.add_axes([0.2, 0.08, 0.6, 0.03])
azim_slider = Slider(azim_ax, 'Azimuth', -180, 180, valinit=-60, valstep=1)

# Elevation slider (vertical tilt)
elev_ax = fig.add_axes([0.2, 0.03, 0.6, 0.03])
elev_slider = Slider(elev_ax, 'Elevation', -90, 90, valinit=25, valstep=1)

def update_view(val):
    ax.view_init(elev=elev_slider.val, azim=azim_slider.val)
    fig.canvas.draw_idle()

azim_slider.on_changed(update_view)
elev_slider.on_changed(update_view)

plt.show()